In [1]:
import torch as t
from torch import Tensor
from jaxtyping import Float

from nnsight import LanguageModel

import sys

sys.path.append("../../../")

from nngine import alter

device = "cuda"

In [2]:
import torch as t
from torch import Tensor
from jaxtyping import Float
from tqdm import tqdm
import numpy as np

from nnsight.models.UnifiedTransformer import UnifiedTransformer

from transformer_lens import HookedTransformer

device = "cuda"

model = UnifiedTransformer(
    'gpt2-small',
    processing=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device=device,
)
tokenizer = model.tokenizer

model.set_use_hook_mlp_in(True)
model.set_use_split_qkv_input(True)
model.set_use_attn_result(True)

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
from ioi_dataset import IOIDataset

N = 25
clean_dataset = IOIDataset(
    prompt_type='mixed',
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=device
)
corr_dataset = clean_dataset.gen_flipped_prompts('ABC->XYZ, BAB->XYZ')

In [4]:
def ave_logit_diff(
    logits: Float[Tensor, 'batch seq d_vocab'],
    ioi_dataset: IOIDataset,
    per_prompt: bool = False
):
    '''
        Return average logit difference between correct and incorrect answers
    '''
    # Get logits for indirect objects
    batch_size = logits.size(0)
    io_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.io_tokenIDs[:batch_size]]
    s_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.s_tokenIDs[:batch_size]]
    # Get logits for subject
    logit_diff = io_logits - s_logits
    return logit_diff if per_prompt else logit_diff.mean()


with t.no_grad():
    # with model.trace(clean_dataset.toks):
    #     clean_logits = model.output.logits.save()

    # with model.trace(corr_dataset.toks):
    #     corrupt_logits = model.output.logits.save()

    with model.trace(clean_dataset.toks):
        clean_logits = model.output.save()

    with model.trace(corr_dataset.toks):
        corrupt_logits = model.output.save()

clean_logits = clean_logits.value
corrupt_logits = corrupt_logits.value

clean_logit_diff = ave_logit_diff(clean_logits, clean_dataset).item()
corrupt_logit_diff = ave_logit_diff(corrupt_logits, corr_dataset).item()

def ioi_metric(
    logits: Float[Tensor, "batch seq_len d_vocab"],
    corrupted_logit_diff: float = corrupt_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
    ioi_dataset: IOIDataset = clean_dataset
 ):
    patched_logit_diff = ave_logit_diff(logits, ioi_dataset)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

def negative_ioi_metric(logits: Float[Tensor, "batch seq_len d_vocab"]):
    return -ioi_metric(logits)
    
# Get clean and corrupt logit differences
with t.no_grad():
    clean_metric = ioi_metric(clean_logits, corrupt_logit_diff, clean_logit_diff, clean_dataset)
    corrupt_metric = ioi_metric(corrupt_logits, corrupt_logit_diff, clean_logit_diff, corr_dataset)

print(f'Clean direction: {clean_logit_diff}, Corrupt direction: {corrupt_logit_diff}')
print(f'Clean metric: {clean_metric}, Corrupt metric: {corrupt_metric}')

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Clean direction: 2.805180311203003, Corrupt direction: 1.5087411403656006
Clean metric: 0.9999999403953552, Corrupt metric: 0.0


In [5]:
nn_model = LanguageModel(
    'openai-community/gpt2',
    device_map="cuda:0",
    dispatch=True,
    tokenizer=model.tokenizer
)

alter(nn_model)

In [6]:
import new_eap

import importlib

importlib.reload(new_eap)

graph = new_eap.EAP({}, components=["head", "mlp"])

In [7]:
graph.run(
    nn_model,
    model.tokenizer,
    clean_dataset.toks,
    corr_dataset.toks,
    batch_size=25,
    metric=ioi_metric,
)

tensor([[[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          ...,
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00]],

         [[ 1.4735e-05, -2.0519e-05, -1.4828e-05,  ...,  3.1798e-05,
            2.3599e-05, -8.6751e-05],
          [ 9.2246e-07, -8.5710e-07, -4.6749e-07,  ..., -8.1646e-07,
           -3.5441e-06, -2.8212e-08],
          [ 5.8993e-04,  2.4503e-04, -8.4640e-05,  ..., -9.8564e-04,
            4.7063e-04, -6.7333e-04],
          ...,
     

In [8]:
edges = graph.top_edges(n=100, format=True)

head.0.3 -> [0.002] -> head.1.9.q
head.0.1 -> [0.002] -> head.1.9.q
head.0.6 -> [0.002] -> head.1.9.v
head.0.6 -> [-0.002] -> head.1.4.v
head.0.6 -> [-0.001] -> head.1.9.q
head.0.9 -> [0.001] -> head.1.4.v
head.0.1 -> [0.001] -> head.11.2.k
head.0.10 -> [0.001] -> head.1.9.q
head.0.4 -> [0.001] -> head.1.10.v
head.0.6 -> [-0.001] -> head.1.8.v
head.0.9 -> [0.001] -> head.1.6.k
head.0.6 -> [0.001] -> head.1.7.v
head.0.6 -> [-0.001] -> head.1.11.v
head.0.9 -> [-0.001] -> head.1.7.v
head.0.5 -> [0.001] -> head.1.9.q
head.0.7 -> [0.001] -> head.4.0.k
head.0.7 -> [-0.001] -> head.1.10.v
head.0.6 -> [0.001] -> head.1.10.v
head.0.9 -> [-0.001] -> head.1.9.q
head.0.11 -> [-0.001] -> head.1.4.v
head.0.9 -> [0.001] -> head.1.6.q
head.1.9 -> [-0.001] -> head.4.0.k
head.0.10 -> [0.001] -> head.1.7.v
head.1.9 -> [0.001] -> head.4.0.q
head.10.0 -> [0.001] -> head.11.2.q
head.0.10 -> [-0.001] -> head.1.8.v
head.0.11 -> [0.001] -> head.1.10.v
head.0.10 -> [-0.001] -> head.1.6.q
head.0.3 -> [0.001] -> 